In [303]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from catboost import CatBoostClassifier
from sklearn.cluster import KMeans

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score

In [304]:
df_train = pd.read_csv('train.csv')
df_test = pd.read_csv('test.csv')

In [305]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39073 entries, 0 to 39072
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   sampleid             39073 non-null  int64 
 1   age                  39073 non-null  int64 
 2   workclass            36849 non-null  object
 3   fnlwgt               39073 non-null  int64 
 4   education            39073 non-null  object
 5   educational_num      39073 non-null  int64 
 6   marital_status       39073 non-null  object
 7   occupation           36839 non-null  object
 8   relationship         39073 non-null  object
 9   race                 39073 non-null  object
 10  gender               39073 non-null  object
 11  capital_gain         39073 non-null  int64 
 12  capital_loss         39073 non-null  int64 
 13  hours_per_week       39073 non-null  int64 
 14  native_country       38392 non-null  object
 15  income               39073 non-null  object
 16  prof

In [306]:
frame_tsk1 = df_train[(df_train['income'] == ">50K") & (df_train['native_country'] != 'United-States')]

In [307]:
frame_tsk1

,sampleid,age,workclass,fnlwgt,education,educational_num,marital_status,occupation,relationship,race,gender,capital_gain,capital_loss,hours_per_week,native_country,income,profile_description
94,13575,34,Self-emp-not-inc,195891,Bachelors,13,Married-civ-spouse,Sales,Husband,White,Male,0,0,50,NaN,>50K,prospectus pitch coordination quarterly stakeh...
122,44546,51,Self-emp-not-inc,120781,Prof-school,15,Married-civ-spouse,Prof-specialty,Husband,Other,Male,99999,0,70,India,>50K,management framework cloud scalable bandwidth ...
161,47115,25,Private,110978,Assoc-acdm,12,Married-civ-spouse,Adm-clerical,Wife,Asian-Pac-Islander,Female,0,0,37,India,>50K,catering development linen banquet analysis
170,2048,65,Private,444725,Prof-school,15,Married-spouse-absent,Craft-repair,Not-in-family,White,Male,0,0,48,Hungary,>50K,rigging management monitoring scaffold soldering
185,17958,64,NaN,168340,HS-grad,9,Married-civ-spouse,NaN,Husband,White,Male,0,0,40,NaN,>50K,housekeeping concierge development coordinatio...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
38643,16934,35,Private,292472,Doctorate,16,Married-civ-spouse,Prof-specialty,Husband,Asian-Pac-Islander,Male,0,0,40,Taiwan,>50K,experience debug database backend framework ma...
38660,29181,41,Private,529216,Bachelors,13,Divorced,Tech-support,Unmarried,Black,Male,7430,0,45,NaN,>50K,database microservice operations backend serve...
38768,31535,31,Private,279015,Masters,14,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,2415,70,Taiwan,>50K,development leverage planning prospectus pitch...
38939,22287,41,Private,182567,Bachelors,13,Married-civ-spouse,Exec-managerial,Husband,White,Male,0,0,40,NaN,>50K,shareholder development pitch margin liquidity...


In [308]:
sub1 = frame_tsk1['native_country'].mode()[0]

In [309]:
sub1

'Philippines'

In [310]:
 total = df_train.groupby('occupation').size()

In [311]:
rata_mare = frame_tsk1.groupby('occupation').size()

In [312]:
rate = rata_mare/total

In [313]:
sub2 = rate.idxmax()

In [314]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39073 entries, 0 to 39072
Data columns (total 17 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   sampleid             39073 non-null  int64 
 1   age                  39073 non-null  int64 
 2   workclass            36849 non-null  object
 3   fnlwgt               39073 non-null  int64 
 4   education            39073 non-null  object
 5   educational_num      39073 non-null  int64 
 6   marital_status       39073 non-null  object
 7   occupation           36839 non-null  object
 8   relationship         39073 non-null  object
 9   race                 39073 non-null  object
 10  gender               39073 non-null  object
 11  capital_gain         39073 non-null  int64 
 12  capital_loss         39073 non-null  int64 
 13  hours_per_week       39073 non-null  int64 
 14  native_country       38392 non-null  object
 15  income               39073 non-null  object
 16  prof

In [315]:
df_train = df_train.drop(['relationship', 'race'], axis=1)
df_test = df_test.drop(['relationship', 'race'], axis=1)

In [316]:
df_test.select_dtypes(include='object').columns

Index(['workclass', 'education', 'marital_status', 'occupation', 'gender',
       'native_country', 'profile_description'],
      dtype='object')

In [317]:
catego =['workclass', 'education', 'marital_status', 'occupation', 'gender',
       'native_country']

In [318]:

df_train = pd.get_dummies(df_train,  columns=catego, drop_first=True)
df_test = pd.get_dummies(df_test, columns=catego, drop_first=True)

In [319]:
for col in df_train.select_dtypes(include='bool'):
    df_train[col] = df_train[col].astype(int)

for col in df_test.select_dtypes(include='bool'):
    df_test[col] = df_test[col].astype(int)



In [320]:
df_train.columns

Index(['sampleid', 'age', 'fnlwgt', 'educational_num', 'capital_gain',
       'capital_loss', 'hours_per_week', 'income', 'profile_description',
       'workclass_Local-gov', 'workclass_Never-worked', 'workclass_Private',
       'workclass_Self-emp-inc', 'workclass_Self-emp-not-inc',
       'workclass_State-gov', 'workclass_Without-pay', 'education_11th',
       'education_12th', 'education_1st-4th', 'education_5th-6th',
       'education_7th-8th', 'education_9th', 'education_Assoc-acdm',
       'education_Assoc-voc', 'education_Bachelors', 'education_Doctorate',
       'education_HS-grad', 'education_Masters', 'education_Preschool',
       'education_Prof-school', 'education_Some-college',
       'marital_status_Married-AF-spouse', 'marital_status_Married-civ-spouse',
       'marital_status_Married-spouse-absent', 'marital_status_Never-married',
       'marital_status_Separated', 'marital_status_Widowed',
       'occupation_Armed-Forces', 'occupation_Craft-repair',
       'occupation_

In [321]:
df_train

,sampleid,age,fnlwgt,educational_num,capital_gain,capital_loss,hours_per_week,income,profile_description,workclass_Local-gov,...,native_country_Portugal,native_country_Puerto-Rico,native_country_Scotland,native_country_South,native_country_Taiwan,native_country_Thailand,native_country_Trinadad&Tobago,native_country_United-States,native_country_Vietnam,native_country_Yugoslavia
0,34343,71,77253,9,0,0,17,<=50K,welding analysis operations timber crane certi...,0,...,0,0,0,0,0,0,0,1,0,0
1,18560,17,329783,6,0,0,10,<=50K,merger KPI leverage management certified profe...,0,...,0,0,0,0,0,0,0,1,0,0
2,12478,27,91257,9,0,0,40,<=50K,support garnish turndown experience concierge,0,...,0,0,0,0,0,0,0,0,0,0
3,561,43,125577,9,0,0,40,<=50K,floral amenity concierge linen training operat...,0,...,0,0,0,0,0,0,0,1,0,0
4,3428,31,137978,13,0,0,40,<=50K,margin monitoring merger maintenance portfolio,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
39068,38074,33,217460,9,0,0,60,>50K,support quarterly forecast analysis profession...,0,...,0,0,0,0,0,0,0,1,0,0
39069,16307,56,216851,13,0,0,40,>50K,governance mandate amendment professional subs...,1,...,0,0,0,0,0,0,0,1,0,0
39070,26861,36,136629,10,0,0,40,>50K,standards rebar professional excavation weldin...,0,...,0,0,0,0,0,0,0,1,0,0
39071,20603,32,80058,9,0,0,40,<=50K,upholstery laundry coordination experience ban...,0,...,0,0,0,0,0,0,0,1,0,0


In [322]:
df_test.columns

Index(['sampleid', 'age', 'fnlwgt', 'educational_num', 'capital_gain',
       'capital_loss', 'hours_per_week', 'profile_description',
       'workclass_Local-gov', 'workclass_Private', 'workclass_Self-emp-inc',
       'workclass_Self-emp-not-inc', 'workclass_State-gov',
       'workclass_Without-pay', 'education_11th', 'education_12th',
       'education_1st-4th', 'education_5th-6th', 'education_7th-8th',
       'education_9th', 'education_Assoc-acdm', 'education_Assoc-voc',
       'education_Bachelors', 'education_Doctorate', 'education_HS-grad',
       'education_Masters', 'education_Preschool', 'education_Prof-school',
       'education_Some-college', 'marital_status_Married-AF-spouse',
       'marital_status_Married-civ-spouse',
       'marital_status_Married-spouse-absent', 'marital_status_Never-married',
       'marital_status_Separated', 'marital_status_Widowed',
       'occupation_Armed-Forces', 'occupation_Craft-repair',
       'occupation_Exec-managerial', 'occupation_Farmin

In [323]:
features = [ 'age', 'fnlwgt', 'educational_num', 'capital_gain',
       'capital_loss', 'hours_per_week', 'profile_description',
       'workclass_Local-gov', 'workclass_Private', 'workclass_Self-emp-inc',
       'workclass_Self-emp-not-inc', 'workclass_State-gov',
       'workclass_Without-pay', 'education_11th', 'education_12th',
       'education_1st-4th', 'education_5th-6th', 'education_7th-8th',
       'education_9th', 'education_Assoc-acdm', 'education_Assoc-voc',
       'education_Bachelors', 'education_Doctorate', 'education_HS-grad',
       'education_Masters', 'education_Preschool', 'education_Prof-school',
       'education_Some-college', 'marital_status_Married-AF-spouse',
       'marital_status_Married-civ-spouse',
       'marital_status_Married-spouse-absent', 'marital_status_Never-married',
       'marital_status_Separated', 'marital_status_Widowed',
       'occupation_Armed-Forces', 'occupation_Craft-repair',
       'occupation_Exec-managerial', 'occupation_Farming-fishing',
       'occupation_Handlers-cleaners', 'occupation_Machine-op-inspct',
       'occupation_Other-service', 'occupation_Priv-house-serv',
       'occupation_Prof-specialty', 'occupation_Protective-serv',
       'occupation_Sales', 'occupation_Tech-support',
       'occupation_Transport-moving', 'gender_Male', 'native_country_Canada',
       'native_country_China', 'native_country_Columbia',
       'native_country_Cuba', 'native_country_Dominican-Republic',
       'native_country_Ecuador', 'native_country_El-Salvador',
       'native_country_England', 'native_country_France',
       'native_country_Germany', 'native_country_Greece',
       'native_country_Guatemala', 'native_country_Haiti',
       'native_country_Honduras', 'native_country_Hong',
       'native_country_Hungary', 'native_country_India', 'native_country_Iran',
       'native_country_Ireland', 'native_country_Italy',
       'native_country_Jamaica', 'native_country_Japan', 'native_country_Laos',
       'native_country_Mexico', 'native_country_Nicaragua',
       'native_country_Outlying-US(Guam-USVI-etc)', 'native_country_Peru',
       'native_country_Philippines', 'native_country_Poland',
       'native_country_Portugal', 'native_country_Puerto-Rico',
       'native_country_Scotland', 'native_country_South',
       'native_country_Taiwan', 'native_country_Thailand',
       'native_country_Trinadad&Tobago', 'native_country_United-States',
       'native_country_Vietnam', 'native_country_Yugoslavia']

x = df_train[features]
y = df_train['income']

x_final = df_test[features]

In [324]:
x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 39073 entries, 0 to 39072
Data columns (total 87 columns):
 #   Column                                     Non-Null Count  Dtype 
---  ------                                     --------------  ----- 
 0   age                                        39073 non-null  int64 
 1   fnlwgt                                     39073 non-null  int64 
 2   educational_num                            39073 non-null  int64 
 3   capital_gain                               39073 non-null  int64 
 4   capital_loss                               39073 non-null  int64 
 5   hours_per_week                             39073 non-null  int64 
 6   profile_description                        39073 non-null  object
 7   workclass_Local-gov                        39073 non-null  int64 
 8   workclass_Private                          39073 non-null  int64 
 9   workclass_Self-emp-inc                     39073 non-null  int64 
 10  workclass_Self-emp-not-inc        

In [325]:
x_train, x_test, y_train, y_test = train_test_split(x, y , random_state=42)

In [326]:
model1 = CatBoostClassifier(iterations=1000, learning_rate=0.1, verbose=100, cat_features=['profile_description'])

In [327]:
model1.fit(x_train, y_train)

0:	learn: 0.5987381	total: 55.6ms	remaining: 55.5s


KeyboardInterrupt: 

In [ ]:
y_pred = model1.predict(x_test)

In [ ]:
y_pred

array(['<=50K', '<=50K', '<=50K', ..., '<=50K', '<=50K', '<=50K'],
      shape=(9769,), dtype=object)

In [328]:
y_test

37272    <=50K
1912     <=50K
27220    <=50K
33245     >50K
27732     >50K
         ...  
10814    <=50K
16831    <=50K
37606    <=50K
3618     <=50K
121      <=50K
Name: income, Length: 9769, dtype: object

In [ ]:
scor = f1_score(y_pred, y_test)

ValueError: pos_label=1 is not a valid label. It should be one of ['<=50K', '>50K']